# Nivel middle — Análisis de trayectorias UAV con B-splines

**Perfil:** estudiantes que conocen variables, condicionales, funciones sencillas y nociones de media, pendiente y derivada.

## Objetivos

Al terminar podrás:

- preparar un dataset con pandas;
- separar varias trayectorias mediante saltos temporales;
- centrar las coordenadas;
- ajustar una B-spline suavizada;
- calcular \(f\), \(f'\) y \(f''\);
- localizar ceros, máximos, mínimos e inflexiones;
- organizar los resultados en tablas.

En este nivel se analizan los tres ejes, pero las raíces se calculan con una búsqueda numérica sobre una malla.

## 1. Instalar bibliotecas

In [ ]:
!pip -q install "numpy>=1.24" "pandas>=2.0" "scipy>=1.8" "matplotlib>=3.7" "datasets>=2.18" "pyarrow>=14"

## 2. Importaciones y parámetros

Los parámetros editables están concentrados en una sola celda.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from scipy.interpolate import UnivariateSpline
from scipy.optimize import brentq
from IPython.display import display

DATASET_ID = "riotu-lab/Synthetic-UAV-Flight-Trajectories"

NUMERO_FILAS = 150_000
TRAYECTORIA = 0
GAP_FACTOR = 5.0
SUAVIZADO_RELATIVO = 0.02
GRID_SIZE = 2000

OUTPUT_DIR = Path("resultados_middle")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Entorno preparado.")

Entorno preparado.


## 3. Descargar las columnas necesarias

Se usa streaming para no cargar el dataset completo.

In [ ]:
columnas = ["timestamp", "tx", "ty", "tz"]

dataset_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
)

filas = (
    {columna: fila[columna] for columna in columnas}
    for fila in dataset_stream.take(NUMERO_FILAS)
)

df = pd.DataFrame(filas)

print(f"Filas descargadas: {len(df):,}")
display(df.head())

README.md:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/5093 [00:00<?, ?it/s]

Filas descargadas: 150,000


,timestamp,tx,ty,tz
0,1.700250e+09,-20.110498,2.792424,17.071516
1,1.700250e+09,-20.027747,2.781007,17.104970
2,1.700250e+09,-19.958962,2.779229,17.148497
3,1.700250e+09,-19.898282,2.804703,17.221119
4,1.700250e+09,-19.857856,2.868688,17.274563


## 4. Limpieza de datos

Convertimos las columnas a valores numéricos y eliminamos valores faltantes o infinitos.

In [ ]:
for columna in columnas:
    df[columna] = pd.to_numeric(df[columna], errors="coerce")

antes = len(df)

df = (
    df.replace([np.inf, -np.inf], np.nan)
      .dropna(subset=columnas)
      .reset_index(drop=True)
)

print(f"Filas eliminadas: {antes - len(df):,}")
display(df.describe())

## 5. Separar trayectorias

Calculamos:

\[
\Delta t_i=t_{i+1}-t_i.
\]

Se inicia otra trayectoria cuando el tiempo deja de crecer o cuando el salto es mucho mayor que el paso temporal habitual.

In [ ]:
def add_trajectory_ids(frame, gap_factor=5.0):
    if gap_factor <= 1:
        raise ValueError("gap_factor debe ser mayor que 1.")

    timestamps = frame["timestamp"].to_numpy(dtype=float)
    dt = np.diff(timestamps)

    positive_dt = dt[np.isfinite(dt) & (dt > 0)]
    if positive_dt.size == 0:
        raise ValueError("No hay incrementos temporales positivos.")

    nominal_dt = float(np.median(positive_dt))
    threshold = gap_factor * nominal_dt

    break_after = (~np.isfinite(dt)) | (dt <= 0) | (dt > threshold)
    starts = np.r_[True, break_after]

    result = frame.copy()
    result["trajectory_id"] = np.cumsum(starts) - 1

    return result, nominal_dt, threshold


df, nominal_dt, threshold = add_trajectory_ids(df, GAP_FACTOR)

sizes = df.groupby("trajectory_id").size().sort_values(ascending=False)
valid_ids = sizes[sizes >= 8].index.to_list()

print(f"Paso temporal típico: {nominal_dt:.6g} s")
print(f"Umbral de separación: {threshold:.6g} s")
print(f"Trayectorias detectadas: {df['trajectory_id'].nunique()}")
display(sizes.head(10).to_frame("n_points"))

## 6. Seleccionar y preparar una trayectoria

`TRAYECTORIA` indica la posición dentro de la lista de trayectorias válidas.

In [ ]:
if not 0 <= TRAYECTORIA < len(valid_ids):
    raise ValueError(
        f"TRAYECTORIA debe estar entre 0 y {len(valid_ids) - 1}."
    )

trajectory_id = valid_ids[TRAYECTORIA]

trajectory = (
    df.loc[
        df["trajectory_id"] == trajectory_id,
        ["timestamp", "tx", "ty", "tz"],
    ]
    .groupby("timestamp", as_index=False)[["tx", "ty", "tz"]]
    .mean()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

trajectory["t"] = (
    trajectory["timestamp"] - trajectory["timestamp"].iloc[0]
)

if len(trajectory) < 8:
    raise ValueError("La trayectoria necesita al menos 8 puntos.")

print(f"ID real: {trajectory_id}")
print(f"Puntos: {len(trajectory)}")
print(f"Duración: {trajectory['t'].iloc[-1]:.3f} s")
display(trajectory.head())

## 7. Centrar las coordenadas

Para cada eje:

\[
f_x(t)=x(t)-\bar{x},\quad
f_y(t)=y(t)-\bar{y},\quad
f_z(t)=z(t)-\bar{z}.
\]

El cero de una función centrada representa la media de la coordenada original.

In [ ]:
source_columns = {"x": "tx", "y": "ty", "z": "tz"}
centered_columns = {"x": "fx", "y": "fy", "z": "fz"}

means = trajectory[["tx", "ty", "tz"]].mean()

for axis, source in source_columns.items():
    trajectory[centered_columns[axis]] = trajectory[source] - means[source]

print("Medias originales:")
display(means.to_frame("mean"))

print("Medias centradas:")
display(trajectory[["fx", "fy", "fz"]].mean().to_frame("centered_mean"))

## 8. Visualización inicial

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot(trajectory["tx"], trajectory["ty"], trajectory["tz"])
ax.scatter(
    trajectory["tx"].iloc[0],
    trajectory["ty"].iloc[0],
    trajectory["tz"].iloc[0],
    label="Inicio",
)
ax.scatter(
    trajectory["tx"].iloc[-1],
    trajectory["ty"].iloc[-1],
    trajectory["tz"].iloc[-1],
    label="Fin",
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title(f"Trayectoria UAV — ID {trajectory_id}")
ax.legend()
plt.show()

trajectory.plot(
    x="t",
    y=["fx", "fy", "fz"],
    figsize=(12, 5),
    grid=True,
    title="Coordenadas centradas",
)
plt.axhline(0, linewidth=1)
plt.show()

## 9. Modelo matemático

Usaremos una spline cúbica suavizada. El parámetro de suavizado se escala con la varianza de cada eje:

\[
s=n\cdot \alpha\cdot \operatorname{Var}(f),
\]

donde \(\alpha\) es `SUAVIZADO_RELATIVO`.

- \(\alpha=0\): interpolación exacta;
- \(\alpha\) pequeño: ajuste cercano;
- \(\alpha\) grande: curva más suave.

In [ ]:
def fit_axis(time_values, centered_values, relative_smoothing=0.02):
    time_values = np.asarray(time_values, dtype=float)
    centered_values = np.asarray(centered_values, dtype=float)

    if len(time_values) < 4:
        raise ValueError("Una spline cúbica necesita al menos 4 puntos.")

    if np.any(np.diff(time_values) <= 0):
        raise ValueError("Los tiempos deben ser estrictamente crecientes.")

    variance = float(np.var(centered_values))
    s_value = len(time_values) * relative_smoothing * variance

    spline = UnivariateSpline(
        time_values,
        centered_values,
        k=3,
        s=s_value,
    )

    return {
        "spline": spline,
        "first": spline.derivative(1),
        "second": spline.derivative(2),
        "s_value": s_value,
    }

## 10. Buscar raíces numéricamente

Primero evaluamos la función en una malla. Cuando dos valores consecutivos tienen signos opuestos, usamos `brentq` para localizar la raíz con mayor precisión.

Este método es fácil de entender, aunque puede perder raíces que apenas tocan cero sin cambiar de signo.

In [ ]:
def deduplicate(values, tolerance):
    result = []

    for value in sorted(values):
        if not result or abs(value - result[-1]) > tolerance:
            result.append(float(value))

    return result


def roots_on_grid(function, start, end, grid_size=2000):
    grid = np.linspace(start, end, grid_size)
    values = np.asarray(function(grid), dtype=float)

    scale = max(float(np.nanmax(np.abs(values))), 1.0)
    zero_tolerance = scale * 1e-9
    roots = []

    for index in range(len(grid) - 1):
        left_t, right_t = grid[index], grid[index + 1]
        left_y, right_y = values[index], values[index + 1]

        if abs(left_y) <= zero_tolerance:
            roots.append(left_t)

        if left_y * right_y < 0:
            root = brentq(function, left_t, right_t)
            roots.append(root)

    if abs(values[-1]) <= zero_tolerance:
        roots.append(grid[-1])

    tolerance = max(end - start, 1.0) * 1e-7
    return deduplicate(roots, tolerance)

## 11. Clasificar máximos, mínimos e inflexiones

Evaluamos el signo inmediatamente a la izquierda y a la derecha de cada raíz.

In [ ]:
def sign_near(function, point, start, end):
    epsilon = max(end - start, 1.0) * 1e-6
    left = max(start, point - epsilon)
    right = min(end, point + epsilon)

    return np.sign(function(left)), np.sign(function(right))


def classify_points(model, start, end):
    derivative_roots = roots_on_grid(
        model["first"], start, end, GRID_SIZE
    )
    second_roots = roots_on_grid(
        model["second"], start, end, GRID_SIZE
    )

    extrema = []
    for root in derivative_roots:
        left_sign, right_sign = sign_near(
            model["first"], root, start, end
        )

        if left_sign > 0 and right_sign < 0:
            point_type = "maximo_local"
        elif left_sign < 0 and right_sign > 0:
            point_type = "minimo_local"
        else:
            continue

        extrema.append({
            "t": root,
            "type": point_type,
            "value": float(model["spline"](root)),
        })

    inflections = []
    for root in second_roots:
        left_sign, right_sign = sign_near(
            model["second"], root, start, end
        )

        if left_sign * right_sign < 0:
            inflections.append({
                "t": root,
                "type": "punto_inflexion",
                "value": float(model["spline"](root)),
            })

    return extrema, inflections

## 12. Analizar los tres ejes

In [ ]:
time_values = trajectory["t"].to_numpy(dtype=float)
start = float(time_values[0])
end = float(time_values[-1])

models = {}
results = {}

for axis in ["x", "y", "z"]:
    values = trajectory[centered_columns[axis]].to_numpy(dtype=float)

    model = fit_axis(
        time_values,
        values,
        relative_smoothing=SUAVIZADO_RELATIVO,
    )

    zeros = roots_on_grid(model["spline"], start, end, GRID_SIZE)
    extrema, inflections = classify_points(model, start, end)

    models[axis] = model
    results[axis] = {
        "zeros": zeros,
        "extrema": extrema,
        "inflections": inflections,
        "s_value": model["s_value"],
    }

for axis, result in results.items():
    maxima = sum(p["type"] == "maximo_local" for p in result["extrema"])
    minima = sum(p["type"] == "minimo_local" for p in result["extrema"])

    print(
        f"Eje {axis}: ceros={len(result['zeros'])}, "
        f"máximos={maxima}, mínimos={minima}, "
        f"inflexiones={len(result['inflections'])}"
    )

## 13. Crear tablas de resultados

In [ ]:
rows = []

for axis, result in results.items():
    for root in result["zeros"]:
        rows.append({
            "axis": axis,
            "type": "cero",
            "t": root,
            "centered_value": 0.0,
        })

    for point in result["extrema"]:
        rows.append({
            "axis": axis,
            "type": point["type"],
            "t": point["t"],
            "centered_value": point["value"],
        })

    for point in result["inflections"]:
        rows.append({
            "axis": axis,
            "type": point["type"],
            "t": point["t"],
            "centered_value": point["value"],
        })

points_df = (
    pd.DataFrame(rows)
      .sort_values(["axis", "t"])
      .reset_index(drop=True)
)

display(points_df.head(30))

## 14. Graficar splines y puntos notables

In [ ]:
dense_time = np.linspace(start, end, GRID_SIZE)
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

for plot_axis, axis in zip(axes, ["x", "y", "z"]):
    model = models[axis]
    result = results[axis]

    plot_axis.scatter(
        time_values,
        trajectory[centered_columns[axis]],
        s=12,
        alpha=0.45,
        label="Datos",
    )
    plot_axis.plot(
        dense_time,
        model["spline"](dense_time),
        linewidth=2,
        label="B-spline",
    )
    plot_axis.axhline(0, linewidth=1)

    zeros = np.asarray(result["zeros"], dtype=float)
    if zeros.size:
        plot_axis.scatter(
            zeros,
            np.zeros_like(zeros),
            s=45,
            marker="o",
            label="Ceros",
        )

    extrema = result["extrema"]
    if extrema:
        t_extrema = np.array([p["t"] for p in extrema])
        y_extrema = np.array([p["value"] for p in extrema])
        plot_axis.scatter(
            t_extrema,
            y_extrema,
            s=60,
            marker="^",
            label="Máximos/mínimos",
        )

    inflections = result["inflections"]
    if inflections:
        t_infl = np.array([p["t"] for p in inflections])
        y_infl = np.array([p["value"] for p in inflections])
        plot_axis.scatter(
            t_infl,
            y_infl,
            s=50,
            marker="s",
            label="Inflexiones",
        )

    plot_axis.set_ylabel(f"f{axis}(t)")
    plot_axis.grid(True, alpha=0.3)
    plot_axis.legend(fontsize=8)

axes[-1].set_xlabel("Tiempo [s]")
fig.suptitle(f"Análisis de la trayectoria {trajectory_id}")
fig.tight_layout()
plt.show()

## 15. Comparar \(f\), \(f'\) y \(f''\)

Cambia `AXIS_TO_EXPLAIN` para analizar otro eje.

In [ ]:
AXIS_TO_EXPLAIN = "x"
model = models[AXIS_TO_EXPLAIN]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(dense_time, model["spline"](dense_time))
axes[0].axhline(0, linewidth=1)
axes[0].set_ylabel("f(t)")
axes[0].set_title(f"Eje {AXIS_TO_EXPLAIN}")

axes[1].plot(dense_time, model["first"](dense_time))
axes[1].axhline(0, linewidth=1)
axes[1].set_ylabel("f'(t)")

axes[2].plot(dense_time, model["second"](dense_time))
axes[2].axhline(0, linewidth=1)
axes[2].set_ylabel("f''(t)")
axes[2].set_xlabel("Tiempo [s]")

for axis_plot in axes:
    axis_plot.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Exportar resultados

In [ ]:
trajectory.to_csv(
    OUTPUT_DIR / "trayectoria_centrada.csv",
    index=False,
)
points_df.to_csv(
    OUTPUT_DIR / "puntos_notables.csv",
    index=False,
)

print("Archivos creados:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path)

## 17. Actividades propuestas

1. Compara `SUAVIZADO_RELATIVO = 0`, `0.02` y `0.2`.
2. Explica por qué el número de extremos cambia con el suavizado.
3. Modifica `roots_on_grid` para registrar también intervalos positivos y negativos.
4. Compara dos trayectorias y redacta una conclusión.
5. Investiga en qué caso una raíz puede existir sin cambio de signo.

### Límite de este nivel

La búsqueda mediante malla es intuitiva, pero no es la opción más rigurosa. El nivel avanzado usa la representación polinómica por tramos de la B-spline y tolerancias numéricas adaptativas.